# EPCLM-007 T4 Screening
Select Tesla T4, enable Internet, and grant access to the GITHUB_TOKEN secret. Save & Run All starts a fresh run and publishes its decision and metrics to GitHub.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import tempfile
import uuid

started=time.monotonic()
stop=started+10500
remote='https://github.com/ArcheLabs/mini-cells.git'
branch='research/epclm007-cell-retrieval-learning'
os.environ['OMP_NUM_THREADS']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'

def run(argv,limit):
    remaining=min(limit,stop-time.monotonic())
    if remaining<=0:
        raise TimeoutError('EPCLM-007 session budget exhausted')
    process=subprocess.Popen(argv,start_new_session=True)
    try:
        code=process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL)
            process.wait()
        raise TimeoutError('EPCLM-007 stopped at session budget. Preserve complete Output.')
    if code:
        raise RuntimeError('EPCLM-007 stopped. Inspect the retained error and checkpoints.')

run_id_suffix=uuid.uuid4().hex[:12]
repo=Path(tempfile.mkdtemp(prefix='minicells-epclm007-'))/'repo'
run(['git','clone','--depth=1','--branch',branch,remote,str(repo)],300)
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base=repo/'research/projects/cellular-language-models/experiments/epclm-007-cell-retrieval-learning'
requirements=base.parent/'epclm-004-unified-memory-10m/m4/requirements.txt'
run_id=f'epclm007-{commit[:12]}-{run_id_suffix}'
out=Path('/kaggle/working')/run_id
sys.path.insert(0,str(repo))
from scripts.research.publish_experiment_results import (
    ExperimentSpec, load_github_token, prepare_artifacts, push_results,
)

load_github_token('GITHUB_TOKEN')
run([sys.executable,'-m','pip','install','-q','-r',str(requirements)],300)
os.chdir(repo)
screen_error=None
try:
    run([sys.executable,str(base/'screen007.py'),str(out),'--started',str(started)],9600)
except (RuntimeError, TimeoutError) as error:
    screen_error=str(error)

out.mkdir(exist_ok=True)
decision_path=out/'decision.json'
if screen_error:
    decision={'status':'FAILED','error':screen_error}
elif decision_path.is_file():
    decision=json.loads(decision_path.read_text())
else:
    decision={'status':'FAILED','error':'Screening exited without decision.json'}
format_id='minicells.epclm007.screening.v1'
result_branch='kaggle/'+run_id
decision.update(format=format_id,run_id=run_id,source_commit=commit,branch=result_branch)
decision_path.write_text(json.dumps(decision,indent=2)+'\n')
print({'screening_status':decision['status'],'decision':str(decision_path)},flush=True)

stage=repo/'results/epclm007-screening'
stage.mkdir(parents=True)
files=[]
for path in sorted(out.rglob('*.json')):
    name=path.relative_to(out)
    if name.parts[0]=='cell':
        continue
    destination=stage/name
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(path,destination)
    files.append(name.as_posix())
spec=ExperimentSpec(stage.relative_to(repo).as_posix(),
                    'artifacts/experiments/epclm007/'+run_id,
                    tuple(files),result_branch,format_id)
try:
    destination=prepare_artifacts(repo,'epclm007',spec,None)
    push_results(repo,destination,'epclm007',result_branch,'GITHUB_TOKEN','origin/main')
    sha=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
    publication={'status':'REMOTE_SHA_VERIFIED','branch':result_branch,'commit':sha}
except Exception:
    (out/'publication.json').write_text(json.dumps({'status':'PUBLICATION_FAILED','branch':result_branch}))
    raise
(out/'publication.json').write_text(json.dumps(publication,indent=2)+'\n')
print(publication,flush=True)
if decision['status']=='FAILED':
    raise RuntimeError(decision['error'])
